In [1]:
import os
import subprocess
import sys
import pandas as pd
import numpy as np

os.makedirs("outputs", exist_ok=True)
if not os.path.exists("agents.csv") or not os.path.exists("agent_transactions.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)

```markdown
# Mobile Money Agent Network Data Quality Audit & Clean-Up

This is a data quality audit and cleaning pipeline for a Mobile Money Agent Network operating across five counties in Kenya.

# Notebook Goals:
1. Audit the unreliable legacy transaction data across 6 data quality dimensions.
2. Design and call a modular cleaning pipeline defined in `cleaning_pipeline.py`.
3. Validate join and referential integrity against the agent registry.
4. Perform county-level and weekly transaction trend aggregation.
5. Document a full **1-Page Data Quality Report** & **Privacy Compliance Statement** (aligned with the **Kenya Data Protection Act 2019**).
```

```markdown
## Create Pipeline Module: `cleaning_pipeline.py`

To keep our production workflow reusable and modular, we package our logic into `cleaning_pipeline.py` which contains `clean_agent_data(txns, agents)` with a valid docstring, returning `(clean_df, log_dict)`.
```

```markdown
## Task 1: Data-Quality Audit

Before executing our cleaning pipelines, we inspect and document the exact state of the raw transactions and agent directories to construct a formal Data-Quality Audit table.
```

In [2]:
# Load raw data for the audit
raw_txns = pd.read_csv("agent_transactions.csv")
raw_agents = pd.read_csv("agents.csv")
audit_txns = raw_txns.drop_duplicates(subset=["txn_id"], keep="first")
registered_agents = set(raw_agents["agent_id"])

numeric_amounts = pd.to_numeric(
    audit_txns["amount"].replace("N/A", np.nan), errors="coerce"
)
numeric_amounts = numeric_amounts.fillna(numeric_amounts.median())
q1, q3 = numeric_amounts.quantile([0.25, 0.75])
iqr = q3 - q1
outlier_count = int(((numeric_amounts < q1 - 1.5 * iqr) | (numeric_amounts > q3 + 1.5 * iqr)).sum())
raw_unknown_count = int((~raw_txns["agent_id"].isin(registered_agents)).sum())
post_dedup_unknown_count = int((~audit_txns["agent_id"].isin(registered_agents)).sum())

# Counts are measured after the same first-record deduplication used by cleaning_pipeline.
audit_data = [
    {
        "dimension": "Uniqueness",
        "issue": "Repeated transaction IDs in the source extract",
        "rows_affected": len(raw_txns) - len(audit_txns),
        "cleaning_action": "Keep the first row for each txn_id",
    },
    {
        "dimension": "Consistency",
        "issue": "Non-standard casing or whitespace in txn_type",
        "rows_affected": int((~audit_txns["txn_type"].isin(["Deposit", "Withdrawal"])).sum()),
        "cleaning_action": "Strip whitespace and standardize transaction-type casing",
    },
    {
        "dimension": "Validity",
        "issue": "Non-ISO date delimiters in txn_date",
        "rows_affected": int(audit_txns["txn_date"].str.contains("/", na=False).sum()),
        "cleaning_action": "Normalize slash-delimited dates and parse as datetime",
    },
    {
        "dimension": "Completeness",
        "issue": "Missing or non-numeric transaction amounts",
        "rows_affected": int(audit_txns["amount"].isna().sum() + (audit_txns["amount"] == "N/A").sum()),
        "cleaning_action": "Median-impute and flag affected amounts",
    },
    {
        "dimension": "Outliers",
        "issue": "Amounts outside the IQR bounds after numeric conversion and imputation",
        "rows_affected": outlier_count,
        "cleaning_action": "Retain transactions and flag is_outlier for review",
    },
    {
        "dimension": "Referential Integrity",
        "issue": f"Unregistered agent rows: {raw_unknown_count} in raw data; {post_dedup_unknown_count} after deduplication",
        "rows_affected": post_dedup_unknown_count,
        "cleaning_action": "Retain unmatched rows and assign county='Unknown'",
    },
]

audit_df = pd.DataFrame(audit_data)
display(audit_df)
print(
    f"Unknown-agent reconciliation: {raw_unknown_count} raw rows -> "
    f"{post_dedup_unknown_count} after transaction-ID deduplication."
)

,dimension,issue,rows_affected,cleaning_action
0,Uniqueness,Repeated transaction IDs in the source extract,12,Keep the first row for each txn_id
1,Consistency,Non-standard casing or whitespace in txn_type,376,Strip whitespace and standardize transaction-t...
2,Validity,Non-ISO date delimiters in txn_date,5,Normalize slash-delimited dates and parse as d...
3,Completeness,Missing or non-numeric transaction amounts,20,Median-impute and flag affected amounts
4,Outliers,Amounts outside the IQR bounds after numeric c...,53,Retain transactions and flag is_outlier for re...
5,Referential Integrity,Unregistered agent rows: 8 in raw data; 7 afte...,7,Retain unmatched rows and assign county='Unknown'


Unknown-agent reconciliation: 8 raw rows -> 7 after transaction-ID deduplication.


```markdown
## Tasks 2-7 & 9: Modular Pipeline Execution & Merging

Now, we import the modular cleaning pipeline function from `cleaning_pipeline.py` to systematically address all data defects detected during our audit phase.
```

In [3]:
from cleaning_pipeline import clean_agent_data

# Execute cleaning pipeline
clean_df, log = clean_agent_data(raw_txns, raw_agents)

print("--- CLEANING PROCESS COMPLETION LOG ---")
for k, v in log.items():
    print(f"{k.replace('_', ' ').capitalize()}: {v}")

# Save clean dataset to outputs/
clean_df.to_csv('outputs/agent_transactions_clean.csv', index=False)
print("\nSuccessfully exported cleaned transactions to 'outputs/agent_transactions_clean.csv'")

--- CLEANING PROCESS COMPLETION LOG ---
Duplicates removed: 12
Imputed values count: 20
Outliers detected: 53
Unknown agents count: 7

Successfully exported cleaned transactions to 'outputs/agent_transactions_clean.csv'


### Cleaning Treatment Summary

1. **Uniqueness:** Removed duplicate `txn_id` rows, retaining the first occurrence (12 removed).
2. **Consistency:** Trimmed whitespace and standardized `txn_type` to `Deposit` or `Withdrawal`.
3. **Validity:** Parsed slash-delimited and ISO dates into a consistent datetime format.
4. **Completeness:** Converted invalid/missing amounts to numeric values, median-imputed 20 records, and kept an `amount_imputed` flag.
5. **Outliers:** Flagged 53 IQR outliers with `is_outlier`; no transactions were discarded solely for being outliers.
6. **Referential integrity:** Eight raw rows reference unregistered `AG099`; one duplicate is removed because the retained first row has a registered agent, leaving seven unmatched transactions. These are retained with `county = 'Unknown'`.

```markdown
## Task 8: Summarise & Aggregates

With our data verified and standard, we perform our aggregations to extract geographic performance and temporal trends.

### A. County-Level Performance Split by Transaction Type
```

In [4]:
county_summary = clean_df.groupby(['county', 'txn_type']).agg(
    total_value=('amount', 'sum'),
    txn_count=('amount', 'count'),
    avg_value=('amount', 'mean')
).reset_index()

display(county_summary)

,county,txn_type,total_value,txn_count,avg_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77050.5,27,2853.722222
2,Kisumu,Deposit,436420.5,74,5897.574324
3,Kisumu,Withdrawal,76374.5,37,2064.175676
4,Mombasa,Deposit,148986.0,72,2069.250000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543590.0,118,4606.694915
7,Nairobi,Withdrawal,162036.0,62,2613.483871
8,Nakuru,Deposit,184890.0,60,3081.500000
9,Nakuru,Withdrawal,328594.5,39,8425.500000


```markdown
### B. Pivot Table of Weekly Transaction Values by County
```

In [5]:
# Aggregate transaction value by county and calendar week
clean_df["week"] = clean_df["txn_date"].dt.to_period("W").astype(str)
weekly_pivot = clean_df.pivot_table(
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0,
).sort_index(axis=1)

display(weekly_pivot)
weekly_pivot.to_csv("outputs/agent_weekly_pivot.csv", index=True, index_label="county")
print(f"Saved outputs/agent_weekly_pivot.csv ({weekly_pivot.shape[0]} counties x {weekly_pivot.shape[1]} weeks).")

week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,267771.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28125.0,27077.0,41077.0,15341.0,29416.0,37439.0,21308.0
Mombasa,6759.0,285669.5,30477.5,25144.5,17050.0,38344.0,47092.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52860.5,43107.5,41373.5,282931.0,56786.5,56432.5,65187.0,30934.5
Nakuru,2743.5,56574.0,28730.0,39795.0,18998.5,18145.0,276609.0,26152.5,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0


Saved outputs/agent_weekly_pivot.csv (6 counties x 10 weeks).


## Task 10: Final Data-Quality Report and Privacy Statement

### Privacy and Data Protection

This analysis uses synthetic transaction records and opaque transaction/agent IDs; it contains no customer names, phone numbers, or other direct customer identifiers. County-level location is retained only for aggregate operational analysis. If real personal data is introduced, processing should have a documented lawful purpose and transparent notice, collect only necessary fields, pseudonymize direct identifiers, restrict access to authorized roles, protect data in transit and at rest, and apply a defined retention/deletion schedule in line with the Kenya Data Protection Act 2019. Unmatched agents and flagged amounts remain visible for controlled review rather than being silently dropped or treated as verified.

The following final report is generated from the audit and cleaning results below; it distinguishes the eight raw unknown-agent rows from the seven retained after transaction-ID deduplication.

In [6]:
final_quality_report = pd.DataFrame(
    [
        {"measure": "Raw transaction rows", "result": len(raw_txns)},
        {"measure": "Duplicate transaction rows removed", "result": log["duplicates_removed"]},
        {"measure": "Clean transaction rows", "result": len(clean_df)},
        {"measure": "Missing amounts imputed", "result": log["imputed_values_count"]},
        {"measure": "IQR outliers flagged (retained)", "result": log["outliers_detected"]},
        {"measure": "Unknown-agent rows in raw source", "result": raw_unknown_count},
        {"measure": "Unknown-agent rows after deduplication", "result": log["unknown_agents_count"]},
        {"measure": "Clean output file", "result": "outputs/agent_transactions_clean.csv"},
        {"measure": "Weekly pivot file", "result": "outputs/agent_weekly_pivot.csv"},
    ]
)
assert post_dedup_unknown_count == log["unknown_agents_count"]
assert int((clean_df["county"] == "Unknown").sum()) == log["unknown_agents_count"]
assert len(clean_df) == len(raw_txns) - log["duplicates_removed"]
display(final_quality_report)

,measure,result
0,Raw transaction rows,612
1,Duplicate transaction rows removed,12
2,Clean transaction rows,600
3,Missing amounts imputed,20
4,IQR outliers flagged (retained),53
5,Unknown-agent rows in raw source,8
6,Unknown-agent rows after deduplication,7
7,Clean output file,outputs/agent_transactions_clean.csv
8,Weekly pivot file,outputs/agent_weekly_pivot.csv
